# Extract Parties from JSON

In [2]:
import pandas as pd
import json
import requests
from concurrent.futures import ThreadPoolExecutor

In [3]:
# Define the years to process
years = [2020, 2021, 2022, 2023, 2024, 2025]

In [4]:
# Function to download JSON data
def download_json(url):
    response = requests.get(url)
    response.raise_for_status()
    return response.json()

In [5]:
allowed = {'ODS','STAN','Piráti','ANO','KDU-ČSL','AUTO','TOP 09','PŘÍSAHA','ABP','Zelení','Stačilo!','SOCDEM','SPD','Svobodní','KSČM'}

all_parties = {}
for year in years:
    url = f"https://zpravy.udh.gov.cz/zpravy/vfz{year}.json"
    data = download_json(url)
    all_parties[year] = [
        p for p in data['parties']
        if p.get('shortName') in allowed
    ]

In [6]:
# Extract parties information
all_extracted = []
for year, parties in all_parties.items():
    for p in parties:
        penizefo_url = next((f['url'] for f in p['files'] if f.get('subject') == 'penizefo'), None)
        bupfo_url = next((f['url'] for f in p['files'] if f.get('subject') == 'bupfo'), None)
        penizepo_url = next((f['url'] for f in p['files'] if f.get('subject') == 'penizepo'), None)
        buppo_url = next((f['url'] for f in p['files'] if f.get('subject') == 'buppo'), None)
        all_extracted.append({
            'year': year,
            'ic': p['ic'],
            'shortName': p['shortName'],
            'longName': p['longName'],
            'penizefo_url': penizefo_url,
            'bupfo_url': bupfo_url,
            'penizepo_url': penizepo_url,
            'buppo_url': buppo_url
        })

df = pd.DataFrame(all_extracted)
df.head(100)

,year,ic,shortName,longName,penizefo_url,bupfo_url,penizepo_url,buppo_url
0,2020,05402450,ABP,Aliance pro budoucnost,https://zpravy.udh.gov.cz/export/vfz2020-oda-p...,https://zpravy.udh.gov.cz/export/vfz2020-oda-b...,https://zpravy.udh.gov.cz/export/vfz2020-oda-p...,https://zpravy.udh.gov.cz/export/vfz2020-oda-b...
1,2020,05989175,AUTO,Motoristé sobě,https://zpravy.udh.gov.cz/export/vfz2020-refer...,https://zpravy.udh.gov.cz/export/vfz2020-refer...,https://zpravy.udh.gov.cz/export/vfz2020-refer...,https://zpravy.udh.gov.cz/export/vfz2020-refer...
2,2020,00442704,KDU-ČSL,KDU-ČSL,https://zpravy.udh.gov.cz/export/vfz2020-kducs...,https://zpravy.udh.gov.cz/export/vfz2020-kducs...,https://zpravy.udh.gov.cz/export/vfz2020-kducs...,https://zpravy.udh.gov.cz/export/vfz2020-kducs...
3,2020,00496936,KSČM,Komunistická strana Čech a Moravy,https://zpravy.udh.gov.cz/export/vfz2020-kscm-...,https://zpravy.udh.gov.cz/export/vfz2020-kscm-...,https://zpravy.udh.gov.cz/export/vfz2020-kscm-...,https://zpravy.udh.gov.cz/export/vfz2020-kscm-...
4,2020,16192656,ODS,Občanská demokratická strana,https://zpravy.udh.gov.cz/export/vfz2020-ods-p...,https://zpravy.udh.gov.cz/export/vfz2020-ods-b...,https://zpravy.udh.gov.cz/export/vfz2020-ods-p...,https://zpravy.udh.gov.cz/export/vfz2020-ods-b...
...,...,...,...,...,...,...,...,...
72,2025,22142631,Stačilo!,Stačilo!,https://zpravy.udh.gov.cz/export/vfz2025-actos...,https://zpravy.udh.gov.cz/export/vfz2025-actos...,https://zpravy.udh.gov.cz/export/vfz2025-actos...,https://zpravy.udh.gov.cz/export/vfz2025-actos...
73,2025,26673908,STAN,STAROSTOVÉ A NEZÁVISLÍ,https://zpravy.udh.gov.cz/export/vfz2025-stan-...,https://zpravy.udh.gov.cz/export/vfz2025-stan-...,https://zpravy.udh.gov.cz/export/vfz2025-stan-...,https://zpravy.udh.gov.cz/export/vfz2025-stan-...
74,2025,71339612,Svobodní,Svobodní,https://zpravy.udh.gov.cz/export/vfz2025-svobo...,https://zpravy.udh.gov.cz/export/vfz2025-svobo...,https://zpravy.udh.gov.cz/export/vfz2025-svobo...,https://zpravy.udh.gov.cz/export/vfz2025-svobo...
75,2025,71339728,TOP 09,TOP 09,https://zpravy.udh.gov.cz/export/vfz2025-top09...,https://zpravy.udh.gov.cz/export/vfz2025-top09...,https://zpravy.udh.gov.cz/export/vfz2025-top09...,https://zpravy.udh.gov.cz/export/vfz2025-top09...


In [7]:
# Collect all donation URLs
donation_tasks = []
for year, parties in all_parties.items():
    for p in parties:
        ic = p['ic']
        for f in p['files']:
            subject = f.get('subject')
            url = f['url']
            if subject in ['penizefo', 'penizepo', 'bupfo', 'buppo']:
                donation_tasks.append((url, ic, year, subject))

# Function to download donation data
def download_donation(task):
    url, ic, year, subject = task
    try:
        data = download_json(url)
        return ic, year, subject, data
    except Exception as e:
        print(f"Error fetching {url}: {e}")
        return ic, year, subject, None

# Download donations in parallel
with ThreadPoolExecutor(max_workers=10) as executor:
    results = list(executor.map(download_donation, donation_tasks))

In [8]:
# Process results
donations_fo = []
donations_po = []
donations_bupfo = []
donations_buppo = []

for ic, year, subject, data in results:
    if data is None:
        continue
    if isinstance(data, list):
        for item in data:
            if subject == 'penizefo':
                donations_fo.append({
                    'year': year,
                    'ic': ic,
                    **item
                })
            elif subject == 'penizepo':
                donations_po.append({
                    'year': year,
                    'ic': ic,
                    **item
                })
            elif subject == 'bupfo':
                donations_bupfo.append({
                    'year': year,
                    'ic': ic,
                    **item
                })
            elif subject == 'buppo':
                donations_buppo.append({
                    'year': year,
                    'ic': ic,
                    **item
                })

In [9]:
df_donations_fo = pd.DataFrame(donations_fo, columns=['year', 'ic','money','lastName','firstName','titleBefore','titleAfter','birthDate','addrCity'])
df_donations_fo.head(5)

,year,ic,money,lastName,firstName,titleBefore,titleAfter,birthDate,addrCity
0,2020,05402450,100000.0,Cváček,Miroslav,,,1969-04-03,
1,2020,05402450,50000.0,Hanus,Adam,,,1984-12-21,
2,2020,05402450,100000.0,Kachlík,Petr,,,1954-06-01,
3,2020,05402450,100000.0,Kachlík,Petr,,,1954-06-01,
4,2020,05402450,100000.0,Kachlík,Tomáš,,,1985-03-19,


In [10]:
# Create DataFrames
df_donations_fo = pd.DataFrame(donations_fo)
df_donations_po = pd.DataFrame(donations_po)
df_donations_bupfo = pd.DataFrame(donations_bupfo)
df_donations_buppo = pd.DataFrame(donations_buppo)

In [11]:
df_donations_po["description"] = "dar"
df_donations_po["donor_type"] = "PO"
df_donations_po = df_donations_po.rename(columns={"money": "value"})
df_donations_po = df_donations_po[['year','ic','value','donor_type','description','companyId','company','date','addrStreet','addrCity','addrZip']]


In [12]:
df_donations_buppo["donor_type"] = "PO"
df_donations_buppo = df_donations_buppo[['year','ic','value','donor_type','description','companyId','company','date','addrStreet','addrCity','addrZip']]


In [13]:
df_donations_bupfo["donor_type"] = "FO"
df_donations_bupfo = df_donations_bupfo[['year','ic','value','donor_type','description','firstName','lastName','titleBefore','titleAfter','birthDate','date','addrCity']]


In [14]:
df_donations_fo["description"] = "dar"
df_donations_fo["donor_type"] = "FO"
df_donations_fo = df_donations_fo.rename(columns={"money": "value"})
df_donations_fo = df_donations_fo[['year','ic','value','donor_type','description','firstName','lastName','titleBefore','titleAfter','birthDate','date','addrCity']]



In [15]:
# Export DataFrames to CSV
df.to_csv('parties.csv', index=False)
df_donations_fo.to_csv('donations_fo.csv', index=False)
df_donations_po.to_csv('donations_po.csv', index=False)
df_donations_bupfo.to_csv('donations_bupfo.csv', index=False)
df_donations_buppo.to_csv('donations_buppo.csv', index=False)

print("All DataFrames exported to CSV files.")

All DataFrames exported to CSV files.


In [16]:
df_donations_fo_bupfo = pd.concat([df_donations_fo, df_donations_bupfo], ignore_index=True)
df_donations_po_buppo = pd.concat([df_donations_po, df_donations_buppo], ignore_index=True)
df_donations_fo_bupfo.to_csv('donations_fo_bupfo.csv', index=False)
df_donations_po_buppo.to_csv('donations_po_buppo.csv', index=False)
